In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
# 1. تحديد الرابط الرئيسي لوظائف تحليل البيانات على موقع مستقل
url = "https://mostaql.com/projects?category=development&keyword=data"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
        " like Gecko) Chrome/115.0.0.0 Safari/537.36"
    )
}

In [3]:
# 2. إرسال الطلب وجلب محتوى الصفحة
response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")
    projects = soup.find_all("tr", class_="project-row")

    scraped_data = []

    # 3. استخراج البيانات من كل مشروع
    for project in projects:
        # 1. المسمى الوظيفي ورابط المشروع
        title_tag = project.find("h2")
        if title_tag and title_tag.find("a"):
            title = title_tag.find("a").text.strip()
            link = title_tag.find("a")["href"]
        else:
            # محاولة بديلة في حال عدم وجود h2
            a_tag = project.find("a")
            title = a_tag.text.strip() if a_tag else "N/A"
            link = a_tag["href"] if a_tag and "href" in a_tag.attrs else "N/A"

        # 2. استخراج الميزانية وعدد العروض من جميع النصوص داخل عنصر المشروع
        # موقع مستقل يضع الميزانية والعروض داخل أوسمة li أو span أو small
        details_text = project.get_text(separator=" ", strip=True)

        budget = "N/A"
        proposals = "N/A"

        # البحث عن الميزانية وعدد العروض من النص الكامل للصف/المشروع
        items = project.find_all(["li", "span", "td"])
        for item in items:
            txt = item.text.strip()
            if "$" in txt or "دولار" in txt:
                budget = txt
            elif "عرض" in txt or "أضف أول عرض" in txt:
                proposals = txt

        scraped_data.append(
            {
                "Platform": "Mostaql",
                "Job_Title": title,
                "Budget": budget,
                "Proposals_Count": proposals,
                "Job_URL": link,
            }
        )

    # 4. تحويل البيانات لـ DataFrame وحفظها في CSV
    df = pd.DataFrame(scraped_data)
    df.to_csv("mostaql_scraped_data.csv", index=False, encoding="utf-8-sig")
    print("تم سحب البيانات وحفظها بنجاح!")
else:
    print(f"فشل الاتصال بالموقع. رمز الحالة: {response.status_code}")

تم سحب البيانات وحفظها بنجاح!


In [4]:
import os

print("مسار الملف على جهازك هو:")
print(os.path.abspath("mostaql_scraped_data.csv"))

مسار الملف على جهازك هو:
c:\Users\A-FF-L4-D10\Desktop\Scraping\mostaql_scraped_data.csv
